In [1]:
# src/sh_ssw_methods/u_anom.py
from __future__ import annotations
import numpy as np
import pandas as pd
import xarray as xr

In [93]:
from datetime import date, timedelta, datetime

In [13]:
from utils.event_iden_funcs import (
    build_events_df
)

In [176]:
from utils.xr_operators import (
    remove_doy_climatology,
)

In [171]:
import os

- detection algorithm

In [353]:
def detect_ssw_u_anom(
    u_daily: xr.DataArray,
    *,
    time_dim: str = "time",
    thres: float = -20.0,          # e.g., -20 (10 hPa), -11 (50 hPa)
    season_month_min: int = 5,     # months condition: (m > 4) & (m < 11) -> 5..10
    season_month_max: int = 10,
    min_gap_days: int = 20,
    persist_days: int = 10,
    level_hpa: float | None = None,
    data_source: str | None = None,
    latitude: str | None = "-60",
) -> tuple[pd.DataFrame, pd.DatetimeIndex]:


    u_anom = remove_doy_climatology(u_daily)

    # To pandas for simple window logic
    t = pd.to_datetime(u_daily[time_dim].to_index())
    s_full = pd.Series(u_daily.values, index=t)
    s_anom = pd.Series(u_anom.values,  index=t)

    mns = t.month
    ndd = len(s_full)

    
    dates_list = []  # to store datetime objects
    u_min1 = []

    for i in range(ndd - 1):
        if (s_anom.iloc[i+1] < thres) & (s_anom.iloc[i] >= thres) & (mns[i+1] > 4) & (mns[i+1] < 11):
            min1 = s_anom.iloc[i-min_gap_days:i].min()
            min2 = s_full.iloc[i:i+persist_days].min()
    
            if (min1 > thres) & (min2 > 0):
                date_obj = t[i+1]
                dates_list.append(date_obj)
                u_min1.append(min1)

    event_dates = pd.to_datetime(dates_list)
    u_anom_event = np.array(u_min1)
    
    # output into df
    events_df = build_events_df(
        dates=event_dates,
        method="u_anom", #### change here the name later
        definition=f"u_anom_{int(thres)}m/s_{int(level_hPa)}hPa_{int(np.abs(latitude))}S",
        data_source=data_source or "",
        threshold=f"{int(thres)}m/s",
        level_hpa=str(level_hpa),
        latitude=str(latitude),
        notes=f"at_least_{persist_days}d_positive_after_ssw; min_gap={min_gap_days}d",
        # extra_cols={
        # "u_anom": u_anom_event, }, 
    )

    return events_df, event_dates

- for 50hPa

In [372]:
thres = -11
level_hpa = 50
latitude = -60

In [373]:
u5060_path =  os.path.abspath(os.path.join(os.getcwd(), "..", ".."))+ "/data/" +"u5060s_era5_1959_2023.nc"
xu5060 = xr.open_dataset(u5060_path)

In [376]:
da = xu5060['u5060S'].sel(time=slice("1979","2020"))

In [377]:
base_start = "1979-01-01"
base_end = "2020-12-31"
time_dim = "time"

In [378]:
u_daily = da.resample({time_dim: "1D"}).mean()

In [379]:
events_df, event_dates = detect_ssw_u_anom(u_daily, thres=thres, level_hpa=level_hpa, latitude=latitude)

In [380]:
event_dates

DatetimeIndex(['1979-10-22', '1980-07-28', '1981-07-28', '1982-10-07',
               '1985-06-12', '1988-09-25', '1994-05-30', '1995-07-14',
               '1996-09-14', '1999-06-16', '2000-10-27', '2002-09-02',
               '2003-10-31', '2004-10-15', '2005-10-08', '2007-07-05',
               '2008-08-04', '2012-10-09', '2013-10-02', '2013-10-30',
               '2014-10-10', '2016-10-18', '2017-10-26', '2019-09-13'],
              dtype='datetime64[ns]', freq=None)

In [381]:
events_df

,date,method,definition,data_source,threshold,level_hpa,latitude,lat_band,notes
0,1979-10-22,u_anom,u_anom_-11m/s_50hPa_60S,,-11m/s,50,-60,<NA>,at_least_10d_positive_after_ssw; min_gap=20d
1,1980-07-28,u_anom,u_anom_-11m/s_50hPa_60S,,-11m/s,50,-60,<NA>,at_least_10d_positive_after_ssw; min_gap=20d
2,1981-07-28,u_anom,u_anom_-11m/s_50hPa_60S,,-11m/s,50,-60,<NA>,at_least_10d_positive_after_ssw; min_gap=20d
3,1982-10-07,u_anom,u_anom_-11m/s_50hPa_60S,,-11m/s,50,-60,<NA>,at_least_10d_positive_after_ssw; min_gap=20d
4,1985-06-12,u_anom,u_anom_-11m/s_50hPa_60S,,-11m/s,50,-60,<NA>,at_least_10d_positive_after_ssw; min_gap=20d
5,1988-09-25,u_anom,u_anom_-11m/s_50hPa_60S,,-11m/s,50,-60,<NA>,at_least_10d_positive_after_ssw; min_gap=20d
6,1994-05-30,u_anom,u_anom_-11m/s_50hPa_60S,,-11m/s,50,-60,<NA>,at_least_10d_positive_after_ssw; min_gap=20d
7,1995-07-14,u_anom,u_anom_-11m/s_50hPa_60S,,-11m/s,50,-60,<NA>,at_least_10d_positive_after_ssw; min_gap=20d
8,1996-09-14,u_anom,u_anom_-11m/s_50hPa_60S,,-11m/s,50,-60,<NA>,at_least_10d_positive_after_ssw; min_gap=20d
9,1999-06-16,u_anom,u_anom_-11m/s_50hPa_60S,,-11m/s,50,-60,<NA>,at_least_10d_positive_after_ssw; min_gap=20d


- repeat for 10hPa

In [390]:
thres = -20
level_hpa = 10
latitude = -60

In [391]:
# open sample total column ozone data
u1060_path =  os.path.abspath(os.path.join(os.getcwd(), "..", ".."))+ "/data/" +"u1060s_era5_1959_2023.nc"
xu1060 = xr.open_dataset(u1060_path)

In [401]:
da = xu1060['u1060S'].sel(time=slice("1979","2021"))

In [402]:
base_start = "1979-01-01"
base_end = "2021-12-31"
time_dim = "time"

In [403]:
u_daily = da.resample({time_dim: "1D"}).mean()

In [404]:
events_df, event_dates = detect_ssw_u_anom(u_daily, thres=thres, level_hpa=level_hpa, latitude=latitude)

In [405]:
event_dates

DatetimeIndex(['1979-10-18', '1982-10-08', '1986-09-06', '1988-08-30',
               '1988-09-25', '1990-09-28', '2000-10-18', '2002-08-22',
               '2004-09-28', '2005-10-08', '2007-09-19', '2008-08-05',
               '2009-07-23', '2012-08-20', '2012-10-08', '2013-09-19',
               '2017-09-20', '2019-09-01'],
              dtype='datetime64[ns]', freq=None)

In [406]:
events_df

,date,method,definition,data_source,threshold,level_hpa,latitude,lat_band,notes
0,1979-10-18,u_anom,u_anom_-20m/s_50hPa_60S,,-20m/s,10,-60,<NA>,at_least_10d_positive_after_ssw; min_gap=20d
1,1982-10-08,u_anom,u_anom_-20m/s_50hPa_60S,,-20m/s,10,-60,<NA>,at_least_10d_positive_after_ssw; min_gap=20d
2,1986-09-06,u_anom,u_anom_-20m/s_50hPa_60S,,-20m/s,10,-60,<NA>,at_least_10d_positive_after_ssw; min_gap=20d
3,1988-08-30,u_anom,u_anom_-20m/s_50hPa_60S,,-20m/s,10,-60,<NA>,at_least_10d_positive_after_ssw; min_gap=20d
4,1988-09-25,u_anom,u_anom_-20m/s_50hPa_60S,,-20m/s,10,-60,<NA>,at_least_10d_positive_after_ssw; min_gap=20d
5,1990-09-28,u_anom,u_anom_-20m/s_50hPa_60S,,-20m/s,10,-60,<NA>,at_least_10d_positive_after_ssw; min_gap=20d
6,2000-10-18,u_anom,u_anom_-20m/s_50hPa_60S,,-20m/s,10,-60,<NA>,at_least_10d_positive_after_ssw; min_gap=20d
7,2002-08-22,u_anom,u_anom_-20m/s_50hPa_60S,,-20m/s,10,-60,<NA>,at_least_10d_positive_after_ssw; min_gap=20d
8,2004-09-28,u_anom,u_anom_-20m/s_50hPa_60S,,-20m/s,10,-60,<NA>,at_least_10d_positive_after_ssw; min_gap=20d
9,2005-10-08,u_anom,u_anom_-20m/s_50hPa_60S,,-20m/s,10,-60,<NA>,at_least_10d_positive_after_ssw; min_gap=20d
